# Exploding Gradient Demonstration (Corrected)
This notebook reliably demonstrates exploding gradients with clear explanations.

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## 2. Set Random Seed

In [ ]:
torch.manual_seed(0)

## 3. Create Synthetic Dataset

In [ ]:
X = torch.randn(100, 10)
y = torch.randn(100, 1)

## 4. Define Deep Neural Network
We intentionally design it to cause exploding gradients:
- Very deep network
- Large weight initialization
- Tanh activation (can amplify gradients)
- No gradient clipping

In [ ]:
class DeepNN(nn.Module):
    def __init__(self):
        super(DeepNN, self).__init__()

        layers = []
        input_size = 10

        for _ in range(50):  # deeper network
            layer = nn.Linear(input_size, 10)

            # VERY large weights → key reason for explosion
            nn.init.normal_(layer.weight, mean=0, std=10)

            layers.append(layer)
            layers.append(nn.Tanh())  # allows gradient flow (no zeroing like ReLU)
            input_size = 10

        layers.append(nn.Linear(10, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## 5. Initialize Model

In [ ]:
model = DeepNN()

## 6. Loss Function and Optimizer
We use a HIGH learning rate to amplify gradient explosion.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

## 7. Forward Pass

In [ ]:
output = model(X)
loss = criterion(output, y)

## 8. Backward Pass
Gradients explode here due to repeated multiplication of large values.

In [ ]:
optimizer.zero_grad()
loss.backward()

## 9. Collect Gradient Values
We expect very large values (possibly 1e3, 1e6 or more).

In [ ]:
gradients = []

for name, param in model.named_parameters():
    if "weight" in name:
        gradients.append(param.grad.abs().mean().item())

print("Gradient values per layer:")
print(gradients)

## 10. Plot Gradients
You should see rapidly increasing values → exploding gradient.

In [ ]:
plt.figure()
plt.plot(gradients, marker='o')
plt.title("Exploding Gradient Across Layers (Corrected)")
plt.xlabel("Layer Index (from input to output)")
plt.ylabel("Average Gradient Magnitude")
plt.show()